# KNN - DUAT

## 1. Imports

In [ ]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)  # mostra a tabela sempre por completo, sem "..."

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors


## 2. Upload dos datasets

In [ ]:
# Upload dos CSVs -- só pede upload se os arquivos ainda não estiverem no ambiente.
# Sem essa checagem, o Colab pede upload TODA vez que a célula é executada, mesmo
# com os dados já presentes (ex: de uma execução anterior na mesma sessão).
try:
    from google.colab import files
    csvs_necessarios = ["dataset_sem_balancear.csv", "dataset_balanceado.csv"]
    if not all(Path(csv).exists() for csv in csvs_necessarios):
        uploaded = files.upload()
except ImportError:
    pass  # fora do Colab: os CSVs já devem estar no diretório de trabalho


In [ ]:
DATASETS = {
    "desbalanceado": pd.read_csv("dataset_sem_balancear.csv"),
    "balanceado": pd.read_csv("dataset_balanceado.csv"),
}

for nome, df in DATASETS.items():
    print(f"{nome}: {df.shape}")


## 3. Configuração

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET = "rotulo"
TEXT_COL = "texto"
NUM_PALAVRAS_COL = "num_palavras"
TYPES_PROPORCAO_COL = "types_proporcao"

# Configuração do embedding TF-IDF combinada entre os 3 modelos (DUAT).
# FIXOS para a comparação ser justa entre KNN, Random Forest e SVM:
#   lowercase, strip_accents, ngram_range=(1, 2) e stopwords em português.
# AJUSTÁVEL por notebook/modelo: max_features e min_df.
EMBEDDING_MAX_FEATURES = 300
EMBEDDING_MIN_DF = 5


def remover_acentos(palavra):
    """Necessário porque o TfidfVectorizer usa strip_accents="unicode" no texto,
    mas não aplica isso à lista de stopwords recebida -- sem isso, "não" na lista
    nunca casa com "nao" (já sem acento) gerado pelo tokenizador, e o sklearn
    emite o warning "Your stop_words may be inconsistent with your preprocessing"."""
    return unicodedata.normalize("NFKD", palavra).encode("ascii", "ignore").decode("utf-8")


STOP_PT = [remover_acentos(palavra) for palavra in stopwords.words("portuguese")]

OUT_DIR = Path("/content/resultados_knn")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 4. Configurações de teste (12 testes)

In [ ]:
# Desenho fatorial, igual nos 3 notebooks: balanceamento x num_palavras x types_proporcao x embedding.
# Regra: types_proporcao só pode ser removida ("sem") quando num_palavras também estiver
# fora ("sem") -- investigamos a hipótese de que types_proporcao é, na prática, um
# "num_palavras" disfarçado, e essa comparação só faz sentido quando num_palavras já
# não está em jogo. Quando num_palavras = "com", types_proporcao é sempre "com".
# Ordem das colunas na tabela final: types_proporcao ao lado de num_palavras, antes de embedding.
TESTS = [
    {"modelo": "KNN", "teste":  1, "balanceamento": "desbalanceado", "num_palavras": "com", "types_proporcao": "com", "embedding": "com"},
    {"modelo": "KNN", "teste":  2, "balanceamento": "desbalanceado", "num_palavras": "sem", "types_proporcao": "com", "embedding": "sem"},
    {"modelo": "KNN", "teste":  3, "balanceamento": "desbalanceado", "num_palavras": "sem", "types_proporcao": "sem", "embedding": "sem"},
    {"modelo": "KNN", "teste":  4, "balanceamento": "desbalanceado", "num_palavras": "sem", "types_proporcao": "com", "embedding": "com"},
    {"modelo": "KNN", "teste":  5, "balanceamento": "desbalanceado", "num_palavras": "sem", "types_proporcao": "sem", "embedding": "com"},
    {"modelo": "KNN", "teste":  6, "balanceamento": "desbalanceado", "num_palavras": "com", "types_proporcao": "com", "embedding": "sem"},
    {"modelo": "KNN", "teste":  7, "balanceamento": "balanceado", "num_palavras": "com", "types_proporcao": "com", "embedding": "com"},
    {"modelo": "KNN", "teste":  8, "balanceamento": "balanceado", "num_palavras": "sem", "types_proporcao": "com", "embedding": "sem"},
    {"modelo": "KNN", "teste":  9, "balanceamento": "balanceado", "num_palavras": "sem", "types_proporcao": "sem", "embedding": "sem"},
    {"modelo": "KNN", "teste": 10, "balanceamento": "balanceado", "num_palavras": "sem", "types_proporcao": "com", "embedding": "com"},
    {"modelo": "KNN", "teste": 11, "balanceamento": "balanceado", "num_palavras": "sem", "types_proporcao": "sem", "embedding": "com"},
    {"modelo": "KNN", "teste": 12, "balanceamento": "balanceado", "num_palavras": "com", "types_proporcao": "com", "embedding": "sem"},
]

# Validação defensiva da regra acima, para não deixar passar uma configuração
# inválida se alguém editar TESTS depois.
for _cfg in TESTS:
    if _cfg["num_palavras"] == "com" and _cfg["types_proporcao"] == "sem":
        raise ValueError(
            f"Configuração inválida no teste {_cfg['teste']}: types_proporcao só pode "
            "ser removida quando num_palavras também estiver fora do treinamento."
        )
print(f"{len(TESTS)} configurações de teste validadas.")


## 5. Pré-processamento (numéricas + embedding TF-IDF combinado)

In [ ]:
def criar_embedding_tfidf(max_features=EMBEDDING_MAX_FEATURES, min_df=EMBEDDING_MIN_DF,
                           ngram_range=(1, 2), stop_words=STOP_PT):
    """
    Configuração de embedding combinada entre os 3 modelos do DUAT.
    Fixos: lowercase, strip_accents, ngram_range, stopwords em português.
    Ajustáveis por notebook: max_features, min_df.
    """
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        stop_words=stop_words,
    )


def criar_preprocessador(config, df):
    numeric_features = [
        col
        for col in df.columns
        if col not in {TARGET, TEXT_COL}
        and (config["num_palavras"] == "com" or col != NUM_PALAVRAS_COL)
        and (config["types_proporcao"] == "com" or col != TYPES_PROPORCAO_COL)
    ]

    transformers = [
        (
            "numericas",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        )
    ]

    if config["embedding"] == "com":
        transformers.append((
            "texto_tfidf",
            criar_embedding_tfidf(
                max_features=config.get("max_features", EMBEDDING_MAX_FEATURES),
                min_df=config.get("min_df", EMBEDDING_MIN_DF),
            ),
            TEXT_COL,
        ))

    return ColumnTransformer(transformers=transformers)


## 6. Função de treino e avaliação
Split 60/20/20 (treino/validação/teste), estratificado. Busca do melhor K entre 1 e 29 (ímpares) no conjunto de validação, calculada de forma eficiente: os vizinhos mais próximos são computados uma única vez e todos os K são avaliados a partir dessa mesma lista.   
Métricas principais (precision/recall/f1) vêm da validação; accuracy no teste é reportada à parte, como checagem final isolada.

In [ ]:
def rodar_teste_knn(config):
    df = DATASETS[config["balanceamento"]]

    x = df.drop(columns=[TARGET]).reset_index(drop=True)
    y = df[TARGET].reset_index(drop=True)

    # Primeiro separa 20% para teste
    x_temp, x_test, y_temp, y_test = train_test_split(
        x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
    )

    # Depois separa 25% dos 80% restantes para validação (25% de 80% = 20% do total)
    x_train, x_val, y_train, y_val = train_test_split(
        x_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp,
    )
    x_train = x_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    x_val = x_val.reset_index(drop=True)
    y_val = y_val.reset_index(drop=True)

    preprocessor = criar_preprocessador(config, df)
    x_train_proc = preprocessor.fit_transform(x_train)
    x_val_proc = preprocessor.transform(x_val)
    x_test_proc = preprocessor.transform(x_test)

    # Busca do melhor K de forma eficiente: calculamos os vizinhos mais próximos UMA
    # VEZ (no conjunto de validação) para o maior K candidato, e avaliamos todos os K
    # a partir dessa mesma lista de vizinhos -- em vez de recriar a busca do zero para
    # cada K, já que a lista de vizinhos por distância não muda com K, só quantos
    # usamos dela para votar.
    k_candidatos = list(range(1, 30, 2))
    k_max = max(k_candidatos)

    vizinhos = NearestNeighbors(n_neighbors=k_max, n_jobs=-1)
    vizinhos.fit(x_train_proc)
    _, indices = vizinhos.kneighbors(x_val_proc)
    rotulos_vizinhos = y_train.to_numpy()[indices]  # shape (n_val, k_max), já ordenado por distância

    f1_por_k = {}
    for k in k_candidatos:
        # voto majoritário entre os k vizinhos mais próximos (rótulo binário 0/1);
        # k é sempre ímpar, então não há empate possível
        preds = (rotulos_vizinhos[:, :k].mean(axis=1) >= 0.5).astype(int)
        f1_por_k[k] = f1_score(y_val, preds, zero_division=0)

    melhor_k = max(f1_por_k, key=f1_por_k.get)

    modelo = KNeighborsClassifier(n_neighbors=melhor_k, n_jobs=-1)
    modelo.fit(x_train_proc, y_train)

    y_val_pred = modelo.predict(x_val_proc)
    y_test_pred = modelo.predict(x_test_proc)
    y_train_pred = modelo.predict(x_train_proc)

    cm_val = confusion_matrix(y_val, y_val_pred)
    report_val = classification_report(y_val, y_val_pred, output_dict=True, zero_division=0)
    report_treino = classification_report(y_train, y_train_pred, output_dict=True, zero_division=0)
    report_teste = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)

    base = {
        **config,
        "melhor_k": melhor_k,
        "linhas_total": len(df),
        "linhas_treino": len(x_train),
        "linhas_validacao": len(x_val),
        "linhas_teste": len(x_test),
    }

    linha_falsa = {
        **base,
        "classe": "falsa (0)",
        "precision": report_val["0"]["precision"],
        "recall": report_val["0"]["recall"],
        "f1_treino": report_treino["0"]["f1-score"],
        "f1_validacao": report_val["0"]["f1-score"],
        "f1_teste": report_teste["0"]["f1-score"],
        "accuracy_treino": accuracy_score(y_train, y_train_pred),
        "accuracy_validacao": accuracy_score(y_val, y_val_pred),
        "accuracy_teste": accuracy_score(y_test, y_test_pred),
        "suporte": int(report_val["0"]["support"]),
        "tp": int(cm_val[0, 0]),
        "tn": int(cm_val[1, 1]),
        "fp": int(cm_val[1, 0]),
        "fn": int(cm_val[0, 1]),
    }

    linha_verdadeira = {
        **base,
        "classe": "verdadeira (1)",
        "precision": report_val["1"]["precision"],
        "recall": report_val["1"]["recall"],
        "f1_treino": report_treino["1"]["f1-score"],
        "f1_validacao": report_val["1"]["f1-score"],
        "f1_teste": report_teste["1"]["f1-score"],
        "accuracy_treino": accuracy_score(y_train, y_train_pred),
        "accuracy_validacao": accuracy_score(y_val, y_val_pred),
        "accuracy_teste": accuracy_score(y_test, y_test_pred),
        "suporte": int(report_val["1"]["support"]),
        "tp": int(cm_val[1, 1]),
        "tn": int(cm_val[0, 0]),
        "fp": int(cm_val[0, 1]),
        "fn": int(cm_val[1, 0]),
    }

    return [linha_falsa, linha_verdadeira]


## 7. Execução dos testes

In [ ]:
# Guarda os resultados por número de teste (não por lista) -- assim, se uma célula de
# teste for reexecutada durante a apresentação, ela apenas substitui o resultado
# daquele teste em vez de duplicar linhas na tabela final.
resultados_por_teste = {}


def rodar_e_mostrar_knn(numero_teste):
    """Roda um teste específico pelo número e mostra as métricas dele na hora
    (mesmas colunas da tabela final) -- pensado para apresentar teste por teste."""
    config = next(c for c in TESTS if c["teste"] == numero_teste)
    linhas = rodar_teste_knn(config)
    resultados_por_teste[numero_teste] = linhas
    return pd.DataFrame(linhas)


**Teste 1** — balanceamento: `desbalanceado` · num_palavras: `com` · types_proporcao: `com` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(1)

**Teste 2** — balanceamento: `desbalanceado` · num_palavras: `sem` · types_proporcao: `com` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(2)

**Teste 3** — balanceamento: `desbalanceado` · num_palavras: `sem` · types_proporcao: `sem` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(3)

**Teste 4** — balanceamento: `desbalanceado` · num_palavras: `sem` · types_proporcao: `com` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(4)

**Teste 5** — balanceamento: `desbalanceado` · num_palavras: `sem` · types_proporcao: `sem` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(5)

**Teste 6** — balanceamento: `desbalanceado` · num_palavras: `com` · types_proporcao: `com` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(6)

**Teste 7** — balanceamento: `balanceado` · num_palavras: `com` · types_proporcao: `com` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(7)

**Teste 8** — balanceamento: `balanceado` · num_palavras: `sem` · types_proporcao: `com` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(8)

**Teste 9** — balanceamento: `balanceado` · num_palavras: `sem` · types_proporcao: `sem` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(9)

**Teste 10** — balanceamento: `balanceado` · num_palavras: `sem` · types_proporcao: `com` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(10)

**Teste 11** — balanceamento: `balanceado` · num_palavras: `sem` · types_proporcao: `sem` · embedding: `com`

In [ ]:
rodar_e_mostrar_knn(11)

**Teste 12** — balanceamento: `balanceado` · num_palavras: `com` · types_proporcao: `com` · embedding: `sem`

In [ ]:
rodar_e_mostrar_knn(12)

## 8. Tabela final com todos os resultados

In [ ]:
resultados = [linha for linhas in resultados_por_teste.values() for linha in linhas]
resultados_df = pd.DataFrame(resultados).sort_values("teste").reset_index(drop=True)
resultados_df


## 9. Visualização comparativa (validação)  
Usar esta para decidir qual configuração é a melhor.

In [ ]:
# Visualização comparativa entre os experimentos: accuracy + F1 de cada rótulo (0 e 1),
# todos calculados sobre o conjunto de VALIDAÇÃO (mesma fonte das métricas da tabela).
# Usar esta versão (validação) para decidir qual configuração é a melhor.
f1_por_classe = resultados_df.pivot(index="teste", columns="classe", values="f1_validacao")
accuracy_por_teste = resultados_df.drop_duplicates("teste").set_index("teste")["accuracy_validacao"]

comparativo = f1_por_classe.join(accuracy_por_teste).sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
x = comparativo.index.astype(str)
ax.plot(x, comparativo["accuracy_validacao"], marker="o", label="Accuracy (validação)")
ax.plot(x, comparativo["falsa (0)"], marker="o", label="F1 (falsa - rótulo 0)")
ax.plot(x, comparativo["verdadeira (1)"], marker="o", label="F1 (verdadeira - rótulo 1)")
ax.set_xlabel("Experimento")
ax.set_ylabel("Score")
ax.set_title("KNN - comparação entre experimentos (validação)")
ax.legend()
plt.tight_layout()
plt.show()


## 10. Visualização comparativa (teste)
Mesmo gráfico, com as métricas do conjunto de teste -- não deve ser usado para escolher a configuração vencedora.

In [ ]:
# Mesma comparação da seção anterior, mas com as métricas do conjunto de TESTE --
# útil para apresentação (mostra o que ocorreu em cada experimento). NÃO usar esta
# versão para escolher a configuração vencedora: essa decisão é papel da validação
# (seção anterior); o teste só deve embasar a confirmação final, depois de decidido.
f1_por_classe_teste = resultados_df.pivot(index="teste", columns="classe", values="f1_teste")
accuracy_por_teste_teste = resultados_df.drop_duplicates("teste").set_index("teste")["accuracy_teste"]

comparativo_teste = f1_por_classe_teste.join(accuracy_por_teste_teste).sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
x = comparativo_teste.index.astype(str)
ax.plot(x, comparativo_teste["accuracy_teste"], marker="o", label="Accuracy (teste)")
ax.plot(x, comparativo_teste["falsa (0)"], marker="o", label="F1 (falsa - rótulo 0)")
ax.plot(x, comparativo_teste["verdadeira (1)"], marker="o", label="F1 (verdadeira - rótulo 1)")
ax.set_xlabel("Experimento")
ax.set_ylabel("Score")
ax.set_title("KNN - comparação entre experimentos (teste)")
ax.legend()
plt.tight_layout()
plt.show()


## 11. Comparação treino vs validação vs teste

In [ ]:
# Comparação treino vs validação vs teste: um gap grande entre as curvas é sinal de
# overfitting (o modelo memorizou o treino em vez de generalizar para dados novos).
comparativo_treino_teste = (
    resultados_df.drop_duplicates("teste")
    .set_index("teste")[["accuracy_treino", "accuracy_validacao", "accuracy_teste"]]
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
x = comparativo_treino_teste.index.astype(str)
ax.plot(x, comparativo_treino_teste["accuracy_treino"], marker="o", label="Accuracy (treino)")
ax.plot(x, comparativo_treino_teste["accuracy_validacao"], marker="o", label="Accuracy (validação)")
ax.plot(x, comparativo_treino_teste["accuracy_teste"], marker="o", label="Accuracy (teste)")
ax.set_xlabel("Experimento")
ax.set_ylabel("Accuracy")
ax.set_title("KNN - treino vs validação vs teste (gap grande indica overfitting)")
ax.legend()
plt.tight_layout()
plt.show()


## 12. Salvar tabela final

In [ ]:
resultado_final_path = OUT_DIR / "resultados_knn.csv"
resultados_df.to_csv(resultado_final_path, index=False)

print("Tabela final salva em:", resultado_final_path)
